Set up the OpenAI client

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Load the data and build the search index

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Create the assistant

In [3]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

Testing it

In [4]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from [https://ollama.com/download](https://ollama.com/download) for your operating system.
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also try:
```bash
curl http://localhost:11434
```

If needed, you can restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about **Olama/Ollama** specifically.\n\nThe closest relevant info is that you **can run the course locally** instead of using Codespaces if you’re comfortable setting up the needed tools like **Python, `uv`, Jupyter, Docker, and any other module-specific tools**. If you do run locally, make sure to **document your setup** and keep it **reproducible**.\n\nIf you meant **Ollama** in the context of running the course locally, that would fall under the same general guidance, but the FAQ here doesn’t give Ollama-specific setup steps.'

The word "Olama" doesn't match "Ollama" in our index. We use lexical search, so it looks for the exact word and finds nothing. The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

This is the limitation of a fixed pipeline. The search runs once with the exact query the user typed, and there's no second chance. The pipeline doesn't know the search failed, so it can't try again with a corrected query.

We need something smarter. We need an agent.

In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — if the course is still open for enrollment, you can usually join.\n\nA few things to check:\n- **Enrollment deadline**: Is registration still open?\n- **Prerequisites**: Do you meet any required background or approvals?\n- **Class capacity**: Is there still space?\n- **Access method**: If it’s an online course, do you need an invite or platform login?\n\nIf you want, I can help you figure out the exact next step if you tell me:\n1. the course name, and  \n2. where you found it (school, platform, company, etc.).'

First we define a top-level search function that queries the index directly. 

In [7]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join registration enrollment late join"}', call_id='call_EMOTocvenU0ARLXuQOFFLBOh', name='search', type='function_call', id='fc_090d8be8652bcd86006a37b54d016c8191b47e27715caf0412', namespace=None, status='completed')

Steps
- 1- Make a call to the LLM with the search tool   <-- (we pay)
- 2- LLM decides to invoke the search('params')
- 3- We invoke the search, we have the results
- 4- Send the results back to the LLM (another call)    <-- (we pay again)
- 5- LLM processes the results
- 6- LLM give the answer
- 7- Send one more API request <-- (we pay again)
- 8- Process and gives the answer

In total we sent x requests to the LLM


In [13]:
import json

args = json.loads(call.arguments)

args

{'query': 'join course discovered course can I join registration enrollment late join'}

In [14]:
call.name

'search'

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

We have to send the entire message history because its the memory of the agent.

In [18]:
messages.append(call)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join registration enrollment late join"}', call_id='call_EMOTocvenU0ARLXuQOFFLBOh', name='search', type='function_call', id='fc_090d8be8652bcd86006a37b54d016c8191b47e27715caf0412', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_EMOTocvenU0ARLXuQOFFLBOh',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get

In [21]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [22]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open. If you’re just learning, you can start anytime.


In [23]:
response.usage

ResponseUsage(input_tokens=772, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=43, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=815)

To calculate the cost, we can use the input tokens, the outout tokens, and the model type.

In [24]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 43)

In [25]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

This is the cost of the second request of the LLM 

In [26]:
# Your tokens
result = calculate_gpt54mini_price(776, 44)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001428


In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [28]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [29]:
for item in response.output:
    print(item)

ResponseFunctionToolCall(arguments='{"query":"join course enroll discover course can I join"}', call_id='call_WqwAsW2gqbJCqTQGUH3H3O6m', name='search', type='function_call', id='fc_0b60e3758f626de9006a37b54f8e0c8191aa7f6f9d91268666', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"course enrollment discover course join after start"}', call_id='call_wRy9Tr7VWtK9j7xhBKwDWWoi', name='search', type='function_call', id='fc_0b60e3758f626de9006a37b54f8e1c8191b95d31c4b98edc0b', namespace=None, status='completed')
ResponseFunctionToolCall(arguments='{"query":"FAQ can I join the course late enrollment"}', call_id='call_ZEkf1KGnUGk0vS1V9Zidu9vh', name='search', type='function_call', id='fc_0b60e3758f626de9006a37b54f8e3481918f06c36fef12daf6', namespace=None, status='completed')


### The Agentic Loop

An agent has three parts:

- 1- Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
- 2- Tools, the functions the agent can call to carry out the task. For us that's only search.
- 3- Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

Everything below is the code that wires these three together inside a loop.

### A function-call helper

We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [30]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [31]:
# add all message histroy
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course enroll discover course can I join"}
function_call: search {"query":"course enrollment discover course join after start"}
function_call: search {"query":"FAQ can I join the course late enrollment"}


In [32]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll discover course can I join"}', call_id='call_WqwAsW2gqbJCqTQGUH3H3O6m', name='search', type='function_call', id='fc_0b60e3758f626de9006a37b54f8e0c8191aa7f6f9d91268666', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment discover course join after start"}', call_id='call_wRy9Tr7VWtK9j7xhBKwDWWoi'

In [33]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...


function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
function_call: search {"query":"course discovered can I join now FAQ enrollment access"}
function_call: search {"query":"joining the course after it started FAQ can I join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and working through the materials right away.

If you want, I can also help you with the next steps for getting started.


In [34]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [36]:
result = agent_loop(instructions, question)

iteration #1...


function_call: search {"query":"join the course discovered course can I join enroll registration access FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and submitting homework as long as the forms/deadlines are still open; registration itself isn’t required for access.

If you want, I can also help you figure out how to start the course or where to find the materials.


In [37]:
result

'Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and submitting homework as long as the forms/deadlines are still open; registration itself isn’t required for access.\n\nIf you want, I can also help you figure out how to start the course or where to find the materials.'

Restricting off-topic questions

In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...


function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
It looks like “queen gambit” is probably not a course/logistics question, and I couldn’t find a relevant FAQ entry for it.

If you meant the chess opening **Queen’s Gambit**, I can’t explain off-topic content here. If you meant something related to the course, feel free to rephrase and I’ll help using the FAQ.

Are there other areas you want to explore?


## ToyAIKit

AI Agent Frameworks (or Agentic Orhcestration Frameworks)

ToyAIKit is for learning, not production work.

In [39]:
!uv add toyaikit

Resolved 127 packages in 31ms
Checked 123 packages in 260ms


In [40]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

We register the search function along with the schema

In [41]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [42]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.

If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

In [43]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Then register it without passing a schema

In [44]:
agent_tools = Tools()
agent_tools.add_tool(search)

You can look at what ToyAIKit produced

In [45]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The chat_interface handles display in the notebook. The callback renders model messages and tool calls as they happen. The runner runs the agent loop, the same while True we wrote by hand. It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.

In [46]:
chat_interface = IPythonChatInterface()

callback = DisplayingRunnerCallback(chat_interface)

In [47]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [49]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


#### This is the total cost of all the calls

In [50]:
result.cost

CostInfo(input_cost=Decimal('0.00271575'), output_cost=Decimal('0.001521'), total_cost=Decimal('0.00423675'))

In [51]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama local run Ollama locally FAQ"}', call_id='call_fHpL

In [52]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received
